In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load Data

In [ ]:
def import_data_from_url(data_url, data_type):
    """
    Imports data from a given URL based on the specified data type.

    Args:
        data_url (str): The URL of the data.
        data_type (str): The type of the data (e.g., 'csv', 'excel').

    Returns:
        pandas.DataFrame: A DataFrame containing the imported data, or None if an error occurs.
    """
    try:
        if data_type.lower() == 'csv':
            df = pd.read_csv(data_url)
        elif data_type.lower() == 'excel':
            df = pd.read_excel(data_url)
        else:
            print(f"Error: Unsupported data type '{data_type}'. Only 'csv' and 'excel' are currently supported.")
            return None
        print(f"Successfully imported {data_type} data from {data_url}")
        return df
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return None

#Aggregate Data at Visit Level

In [ ]:
def aggregate_at_visit_level(df):

  #calculate the spend for each item
  df['Spend'] = df['Quantity']*df['UnitPrice']

  #aggregate the data at visit level
  df_visit = df.groupby(['InvoiceNo','CustomerID']).agg(
      VisitDate = ('InvoiceDate','min'),
      Quantity = ('Quantity','sum'),
      Spend = ('Spend','sum'),
      ProductType = ('StockCode','nunique')
  ).reset_index()

  #rename the columns
  df_visit.columns = ['InvoiceNo','CustomerID','VisitDate','Quantity','Spend','ProductType']

  return df_visit

#Calculate the mean and standard deviation of time interval between purchases for each customer

In [ ]:
def calculate_customer_interval_stats(df):
    """
    Calculates the mean and standard deviation of time intervals between
    consecutive purchases for each customer.

    Args:
        df (pd.DataFrame): DataFrame with visit-level data,
                           must contain 'CustomerID' and 'VisitDate' columns.

    Returns:
        pd.DataFrame: A DataFrame with CustomerID, mean_interval, and std_interval.
    """
    df_sorted = df.sort_values(by=['CustomerID', 'VisitDate'])
    df_sorted['TimeDiff'] = df_sorted.groupby('CustomerID')['VisitDate'].diff().dt.days

    # Filter out rows where 'TimeDiff' is NaN before grouping
    # This ensures only valid intervals are used for aggregation and removes customers
    # with only one purchase (who would have a NaN TimeDiff).
    df_filtered_intervals = df_sorted.dropna(subset=['TimeDiff'])

    customer_interval_stats = df_filtered_intervals.groupby('CustomerID')['TimeDiff'].agg(
        mean_interval='mean',
        std_interval='std'
    ).reset_index()

    return customer_interval_stats

Generate start and end date of the observation and label window

In [ ]:
def test_train_folds(start_date, obs_duration, gap_duration, label_duration, num_folds):
  folds = []
  for i in range(num_folds):
    obs_start = start_date + pd.DateOffset(months=i)
    obs_end = obs_start + obs_duration-pd.Timedelta(days=1)

    gap_start = obs_end + pd.DateOffset(days=1)
    gap_end = gap_start + gap_duration-pd.Timedelta(days=1)

    label_start = gap_end + pd.DateOffset(days=1)
    label_end = label_start + label_duration-pd.Timedelta(days=1)

    folds.append({'fold': i+1, 'obs': (obs_start, obs_end), 'label': (label_start, label_end)})

  for f in folds:
    print(f"Fold {f['fold']}: Obs {f['obs'][0].date()} to {f['obs'][1].date()} | Label {f['label'][0].date()} to {f['label'][1].date()}")
    return folds

#Remove One-hit Wonders

In [ ]:
def remove_one_hit_wonders(df_visit):
  customer_visit_counts = df_visit.groupby('CustomerID')['InvoiceNo'].nunique().reset_index(name='visit_count')
  one_hit_wonders = customer_visit_counts[customer_visit_counts == 1]['CustomerID']
  df_multiple_visits = df_visit[~df_visit['CustomerID'].isin(one_hit_wonders)]

  return df_multiple_visits

#Filter Data in the specified observation/labeling window

In [ ]:
def filter_data_in_fold(df_multiple_visits, start_date, end_date):
  #Convert dates to datetime obj
  start_dt = pd.to_datetime(start_date)
  end_dt = pd.to_datetime(end_date)

  #filter df_multiple_visits for activities within the specified fold
  df_in_fold = df_multiple_visits[(df_multiple_visits['VisitDate'] >= start_dt) & (df_multiple_visits['VisitDate'] <= end_dt)].copy()
  return df_in_fold

#Calculate Recency, Frequency, Monetary, Average Spend, Average Basket size, and Unique type of products

In [ ]:
def calculate_RFM(end_date,df_in_fold):

  end_dt = pd.to_datetime(end_date)
  reference_date = end_dt + pd.Timedelta(days=1)

  #calculate RFM metrics
  rfm_metrics = df_in_fold.groupby('CustomerID').agg(
      Recency = ('VisitDate', lambda x: (reference_date - x.max()).days),
      Frequency = ('InvoiceNo', 'nunique'),
      Monetary = ('Spend', 'sum'),
      AverageSpend = ('Spend', 'mean'),
      BasketSize = ('Quantity', 'mean'),
      ProductType = ('ProductType', 'nunique')
  ).reset_index()

  return rfm_metrics

#Calculate the frequency for the last 30 days till the end_date of the fold

In [ ]:
def calculate_velocity_Ratio(end_date,df_multiple_visit):
  end_dt = pd.to_datetime(end_date)
  start_dt = end_dt - pd.Timedelta(days=30)

  df_in_period = df_multiple_visit[(df_multiple_visit['VisitDate'] >= start_dt) & (df_multiple_visits['VisitDate'] <= end_dt)].copy()

  # calculate visit frequency during the past 30 days
  df_frequency_30 = df_in_period.groupby('CustomerID').agg(
      Frequency_30 = ('InvoiceNo', 'nunique')
  ).reset_index()

  return df_frequency_30

#Calculate the percentage of dominant category among the total spend for each customer

In [ ]:
def calculate_category_dominance(df_in_fold,df_withID_in_fold):
  # Calculate the total spend for each customer
  total_spend = df_in_fold.groupby('CustomerID')['Spend'].sum().reset_index()
  total_spend.rename(columns={'Spend':'GrandTotal'},inplace=True)

  #calculate total spend under each category for each customer
  df_withID_in_fold['Spend']=df_withID_in_fold['Quantity']*df_withID_in_fold['UnitPrice']
  category_spend = df_withID_in_fold.groupby(['CustomerID', 'StockCode'])['Spend'].sum().reset_index()

  df_grand_category_spend =pd.merge(total_spend,category_spend,on='CustomerID', how='left')
  df_grand_category_spend['CategoryRatio'] = df_grand_category_spend['Spend']/df_grand_category_spend['GrandTotal']

  #fill NaN with 0
  df_grand_category_spend['CategoryRatio'] = df_grand_category_spend['CategoryRatio'].fillna(0)

  #calculate the maximum category ratio
  dominance_df = df_grand_category_spend.groupby('CustomerID')['CategoryRatio'].max().reset_index()

  return dominance_df

#Determine customer churn status

In [ ]:
def determine_customer_churn_status(df_train_label_fold1,df_train_obs_fold1):
  #count visits for each customer in obs fold
  label_customer_visit_counts = df_train_label_fold1.groupby('CustomerID').agg(Label_Visits=('InvoiceNo','nunique')).reset_index()

  #Merge customer visit count in label window into obs window in the same fold
  df_train_fold1 = pd.merge(df_train_obs_fold1,label_customer_visit_counts,on='CustomerID',how='left')

  #Fill NaN visits (customers with no transactions in the label window)with 0
  df_train_fold1['Label_Visits'] = df_train_fold1['Label_Visits'].fillna(0).astype(int)

  #Label customer as 'Churned' () or 'Not Churned'
  df_train_fold1['Churn'] = np.where(df_train_fold1['Label_Visits'] == 0, True, False)

  return df_train_fold1

#Modeling

#Model Training

In [ ]:
def get_ready_for_X_Y_train(df):

  # Define features (X) and target (y) for the training set
  feature_columns = df.columns[:-2].tolist()
  X = df[feature_columns]
  Y = df['Churn']

  return X,Y

In [ ]:
def feature_scaling(X):
  #features needs to be scaled for logisitic regression
  #Initialize the StandardScaler
  scaler = StandardScaler()

  #Fit the scaler on the training data and transform it
  X_scaled = scaler.fit_transform(X)

  #Convert the scaled features back to a DataFrame for better readability and to retain column names
  X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=X.index)

  return X_scaled_df

#Logistic Regression (what should this function return?)

In [ ]:
def train_logistic_regression(X_train_scaled_df, Y_train):
  """
    Fit a Logistic Regression model on the scaled training data.

    Args:
        X_train_scaled_df (pd.DataFrame): Scaled features for the training set.
        Y_train (pd.Series): Target variable for the training set.

    Returns:

  """
  # Initialize the Logistic Regression model
  log_reg_model = LogisticRegression(random_state=42, solver='liblinear') # liblinear is good for small datasets and binary classification

  # Train the model on the scaled training data
  log_reg_model.fit(X_train_scaled_df, y_train)

  print("Logistic Regression model trained successfully.")

  # Predict on the training data
  y_pred_train = log_reg_model.predict(X_train_scaled_df)

  print("\n--- Model Evaluation on Training Data ---")

  # Display Classification Report
  print("\nClassification Report:")
  print(classification_report(y_train, y_pred_train))

  # Display Confusion Matrix
  print("\nConfusion Matrix:")
  cm_train = confusion_matrix(y_train, y_pred_train)
  display(pd.DataFrame(cm_train, index=['Actual Churned', 'Actual Not Churned'], columns=['Predicted Churned', 'Predicted Not Churned']))

#Random Forest

In [ ]:
def train_random_forest(X_train_df, Y_train, d, n):
  """
    Fit a Random Forest model on the scaled training data.

    Args:
        X_train_df (pd.DataFrame): features for the training set.
        Y_train (pd.Series): Target variable for the training set.
        d (int): The maximum depth of the tree.
        n (int): The number of trees in the forest.

    Returns:

  """
  # Initialize the Random Forest model
  rf_model = RandomForestClassifier(random_state=42, max_depth= d, n_estimator =n, class_weight='balanced')

  # Train the model on the scaled training data
  rf_model.fit(X_train_df, Y_train)

  # Predict on the unscaled training data
  y_pred_train_rf = rf_model.predict(X_train_df)

  print(f"\n--- Model Evaluation on Training Data (max_depth={d}, n_estimators={n}) ---")
  print("\nClassification Report:")
  print(classification_report(Y_train, y_pred_train_rf))
  print("\nConfusion Matrix:")
  cm_train_rf_md5_ne100 = confusion_matrix(Y_train, y_pred_train_rf)
  display(pd.DataFrame(cm_train_rf_md5_ne100, index=['Actual Churned', 'Actual Not Churned'], columns=['Predicted Churned', 'Predicted Not Churned']))


_______________________________________________

__________________________________________

# Load UCI Online Retail  and UCI Online Retail II

In [ ]:
# Define the URL for the UCI Online Retail dataset
online_retail_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'

# Import the data using the previously defined function
online_retail_df = import_data_from_url(online_retail_url, 'excel')

# Display the first 5 rows if data was successfully loaded
if online_retail_df is not None:
    display(online_retail_df.head())

Successfully imported excel data from https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
# Load the uploaded UCI Online Retail II data from the local file system
uci_retail_data_path = '/content/online_retail_II.xlsx'

# Import the data using the previously defined function
online_retail_ii_df = import_data_from_url(uci_retail_data_path, 'excel')

# Rename 'Customer ID' to 'CustomerID', 'Invoice' to 'InvoiceNo' and 'Price' to 'UnitPrice' for consistency, if they exist
if online_retail_ii_df is not None:
    if 'Customer ID' in online_retail_ii_df.columns:
        online_retail_ii_df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)
    if 'Invoice' in online_retail_ii_df.columns:
        online_retail_ii_df.rename(columns={'Invoice': 'InvoiceNo'}, inplace=True)
    if 'Price' in online_retail_ii_df.columns:
        online_retail_ii_df.rename(columns={'Price': 'UnitPrice'}, inplace=True)

# Display the first 5 rows if data was successfully loaded
if online_retail_ii_df is not None:
    display(online_retail_ii_df.head())

Successfully imported excel data from /content/online_retail_II.xlsx


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


#Concatenate Online Retail and Online Retail II and Remove Duplicates

In [ ]:
# Concatenate the two DataFrames
# Ensure both are not None before concatenating
if online_retail_df is not None and online_retail_ii_df is not None:
    retail_df = pd.concat([online_retail_df, online_retail_ii_df], ignore_index=True)
    print(f"Combined DataFrame shape before dropping duplicates: {retail_df.shape}")

    # Remove duplicate rows
    retail_df.drop_duplicates(inplace=True)
    print(f"Combined DataFrame shape after dropping duplicates: {retail_df.shape}")
else:
    print("One or both DataFrames are None. Cannot concatenate.")

Combined DataFrame shape before dropping duplicates: (1067370, 8)
Combined DataFrame shape after dropping duplicates: (1033035, 8)


#Remove rows with missing customerID

In [ ]:
if retail_df is not None:
    df_withID = retail_df.dropna(subset=['CustomerID']).copy()
    print(f"Shape of DataFrame after dropping rows with missing CustomerID: {df_withID.shape}")
else:
    print("retail_df is None or empty. Cannot proceed with dropna.")
    df_withID = pd.DataFrame() # Initialize an empty DataFrame to prevent further errors

Shape of DataFrame after dropping rows with missing CustomerID: (797884, 8)


#Remove Canceled Invoice

In [ ]:
if not df_withID.empty:
    # Remove rows where 'InvoiceNo' starts with 'C' (canceled orders)
    initial_shape = df_withID.shape[0]
    df_withID_no_cancel = df_withID[~df_withID['InvoiceNo'].astype(str).str.startswith('C')].copy()
    print(f"Shape of DataFrame before removing canceled orders: ({initial_shape}, {df_withID_no_cancel.shape[1]})")
    print(f"Shape of DataFrame after removing canceled orders: {df_withID_no_cancel.shape}")
else:
    print("df_withID is empty. Cannot remove canceled orders.")

Shape of DataFrame before removing canceled orders: (797884, 8)
Shape of DataFrame after removing canceled orders: (779494, 8)


#Aggregate Data at Visit Level

In [ ]:
if not df_withID_no_cancel.empty:
    df_visit = aggregate_at_visit_level(df_withID_no_cancel)
    display(df_visit.head())
else:
    print("df_withID_no_cancel is empty. Cannot aggregate at visit level.")
    df_visit = pd.DataFrame()

,InvoiceNo,CustomerID,VisitDate,Quantity,Spend,ProductType
0,489434,13085.0,2009-12-01 07:45:00,166,505.30,8
1,489435,13085.0,2009-12-01 07:46:00,60,145.80,4
2,489436,13078.0,2009-12-01 09:06:00,193,630.33,19
3,489437,15362.0,2009-12-01 09:08:00,145,310.75,23
4,489438,18102.0,2009-12-01 09:24:00,826,2286.24,17


#Remove one hit wonders

In [ ]:
df_multiple_visits= remove_one_hit_wonders(df_visit)
display(df_multiple_visits.head())

,InvoiceNo,CustomerID,VisitDate,Quantity,Spend,ProductType
0,489434,13085.0,2009-12-01 07:45:00,166,505.30,8
1,489435,13085.0,2009-12-01 07:46:00,60,145.80,4
2,489436,13078.0,2009-12-01 09:06:00,193,630.33,19
3,489437,15362.0,2009-12-01 09:08:00,145,310.75,23
4,489438,18102.0,2009-12-01 09:24:00,826,2286.24,17


#Define train/test observation/label fold

In [ ]:
train_start_date = pd.to_datetime('2009-12-01')
test_start_date = pd.to_datetime('2010-11-01')

obs_duration = pd.DateOffset(months=6)
gap_duration = pd.DateOffset(months=1)
label_duration = pd.DateOffset(months=4)

num_folds = 3

train_folds = test_train_folds(train_start_date, obs_duration, gap_duration, label_duration, num_folds)
test_folds = test_train_folds(test_start_date, obs_duration, gap_duration, label_duration, num_folds)


Fold 1: Obs 2009-12-01 to 2010-05-31 | Label 2010-07-01 to 2010-10-31
Fold 1: Obs 2010-11-01 to 2011-04-30 | Label 2011-06-01 to 2011-09-30


In [ ]:
for i, (train_f, test_f) in enumerate(zip(train_folds, test_folds)):
    print(f"\n--- Fold {i+1} ---")
    print(train_f['obs'])
    print(train_f['label'])
    print(test_f['obs'])
    print(test_f['label'])


--- Fold 1 ---
(Timestamp('2009-12-01 00:00:00'), Timestamp('2010-05-31 00:00:00'))
(Timestamp('2010-07-01 00:00:00'), Timestamp('2010-10-31 00:00:00'))
(Timestamp('2010-11-01 00:00:00'), Timestamp('2011-04-30 00:00:00'))
(Timestamp('2011-06-01 00:00:00'), Timestamp('2011-09-30 00:00:00'))

--- Fold 2 ---
(Timestamp('2010-01-01 00:00:00'), Timestamp('2010-06-30 00:00:00'))
(Timestamp('2010-08-01 00:00:00'), Timestamp('2010-11-30 00:00:00'))
(Timestamp('2010-12-01 00:00:00'), Timestamp('2011-05-31 00:00:00'))
(Timestamp('2011-07-01 00:00:00'), Timestamp('2011-10-31 00:00:00'))

--- Fold 3 ---
(Timestamp('2010-02-01 00:00:00'), Timestamp('2010-07-31 00:00:00'))
(Timestamp('2010-09-01 00:00:00'), Timestamp('2010-12-31 00:00:00'))
(Timestamp('2011-01-01 00:00:00'), Timestamp('2011-06-30 00:00:00'))
(Timestamp('2011-08-01 00:00:00'), Timestamp('2011-11-30 00:00:00'))


#Calculate all the features

In [ ]:
def get_ready_for_features(start_date,end_date,df_multiple_visits,df_withID_no_cancel):
  #filter data in specified
  df_in_fold = filter_data_in_fold(df_multiple_visits, start_date, end_date)

  #calculate RMF in specified period
  rfm_metrics = calculate_RFM(end_date,df_in_fold)

  #calculate CV in specified period
  customer_stats_df = calculate_customer_interval_stats(df_in_fold)
  customer_stats_df['CV'] = customer_stats_df['std_interval']/customer_stats_df['mean_interval']

  #merge CV into rfm
  df_obs = pd.merge(rfm_metrics,customer_stats_df[['CustomerID','CV']],on='CustomerID',how='left')
  # Fill NaN values of 'CV' with -1, which represents that the customer only visit once in the observation window
  df_obs['CV'] = df_obs['CV'].fillna(-1)

  #calculate order frequency for the last 30 days in the specified period
  df_frequency_30 = calculate_velocity_Ratio(end_date,df_multiple_visits)
  df_obs = pd.merge(df_obs,df_frequency_30,on='CustomerID',how='left')

  #calculate velocity ratio
  df_obs['Frequency_30'] = df_obs['Frequency_30'].fillna(0).astype(int)
  df_obs['Velocity_Ratio']=df_obs['Frequency_30']/df_obs['Frequency']

  #aggregate item level data at stockcode per customer
  df_withID_no_cancel.rename(columns={'InvoiceDate': 'VisitDate'}, inplace=True)
  df_withID_in_fold = filter_data_in_fold(df_withID_no_cancel, start_date, end_date)

  #calculate category dominance
  dominance_df = calculate_category_dominance(df_in_fold,df_withID_in_fold)
  df_obs = df_obs.merge(dominance_df,on='CustomerID',how='left')

  return df_obs

#Model Training and Comparing

# Create complete training/testing dataset

In [ ]:
def create_complete_dataset_for_modeling(fold, df_multiple_visits, df_withID_no_cancel):

  """
    Create a complete dataset for modeling, combiing features and labels.
    It can be used for both training and testing.

    Args:
        fold (dict): A dictionary containing 'obs' (observation window) and 'label' (label window) dates.
        df_multiple_visits (pd.DataFrame): DataFrame containing customer visit data, after removing one-hit wonders.
        df_withID_no_cancel (pd.DataFrame): The main retail DataFrame after cleaning (removing missing CustomerIDs and canceled orders).

    Returns:
        pd.DataFrame: A DataFrame with features and a 'Churn' label for the specified fold.
  """


  # 1. extract the start and end date for observation and label fold
  obs_start, obs_end = fold['obs']
  label_start, label_end = fold['label']

  # 获取训练集特征 (obs 期间)
  df_obs = get_ready_for_features(obs_start, obs_end, df_multiple_visits, df_withID_no_cancel)
  # 获取训练集标签 (label 期间)
  df_label = filter_data_in_fold(df_multiple_visits, label_start, label_end)
  # 组合成完整的训练集 DataFrame 并打上 Churn 标签
  df_combined = determine_customer_churn_status(df_label, df_obs)
  return df_combined

#Logistic Regression

In [ ]:
def logistic_regression(fold_id, X_train, Y_train, X_test, Y_test, c):
  """
    Train a Logistic Regression model and evaluate its performance.

    Args:
        fold_id (int): The fold number for logging.
        X_train (pd.DataFrame): Features for the training set.
        Y_train (pd.Series): Target variable for the training set.
        X_test (pd.DataFrame): Features for the testing set.
        Y_test (pd.Series): Target variable for the testing set.
        c (float): Regularization parameter for the Logistic Regression model.

    Returns:
        dict: A dictionary containing ROC_AUC, F1-Score, Precision, and Recall
              for both training and testing data.
  """
  #feature scaling: Logistic regression is sensitive to the scale of features
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  # Initialize the Logistic Regression model
  log_reg_model = LogisticRegression(C=c,random_state=42, solver='liblinear', class_weight='balanced') # Adding class_weight for imbalanced data

  # Train the model on the scaled training data
  log_reg_model.fit(X_train_scaled, Y_train)

  print("Logistic Regression model trained successfully.")

  # Predict on the training data
  y_pred_train = log_reg_model.predict(X_train_scaled)
  y_pred_proba_train = log_reg_model.predict_proba(X_train_scaled)[:, 1]

  # Predict on the test data
  y_pred_test = log_reg_model.predict(X_test_scaled)
  y_pred_proba_test = log_reg_model.predict_proba(X_test_scaled)[:, 1]

  # Calculate evaluation metrics
  from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

  # Training Set Metrics
  auc_train = roc_auc_score(Y_train, y_pred_proba_train)
  f1_train = f1_score(Y_train, y_pred_train)
  precision_train = precision_score(Y_train, y_pred_train)
  recall_train = recall_score(Y_train, y_pred_train)

  # Test Set Metrics
  auc_test = roc_auc_score(Y_test, y_pred_proba_test)
  f1_test = f1_score(Y_test, y_pred_test)
  precision_test = precision_score(Y_test, y_pred_test)
  recall_test = recall_score(Y_test, y_pred_test)

  results = {
      'Fold': fold_id,
      'Model': 'Logistic Regression',
      'Parameters': f'C={c}',
      'ROC_AUC_Train': auc_train,
      'F1_Score_Train': f1_train,
      'Precision_Train': precision_train,
      'Recall_Train': recall_train,
      'ROC_AUC_Test': auc_test,
      'F1_Score_Test': f1_test,
      'Precision_Test': precision_test,
      'Recall_Test': recall_test
  }

  return results

# Random Forest

In [ ]:
def random_forest(fold_id, X_train, Y_train, X_test, Y_test, d, n):
  """
    Train a Random Forest model and evaluate its performance.

    Args:
        fold_id (int): The fold number for logging.
        X_train (pd.DataFrame): Features for the training set.
        Y_train (pd.Series): Target variable for the training set.
        X_test (pd.DataFrame): Features for the testing set.
        Y_test (pd.Series): Target variable for the testing set.
        d (int): The maximum depth of the tree.
        n (int): The number of trees in the forest.

    Returns:
        dict: A dictionary containing ROC_AUC, F1-Score, Precision, and Recall
              for both training and testing data.
  """
  # Initialize the Random Forest model
  rf_model = RandomForestClassifier(random_state=42, max_depth= d, n_estimators =n, class_weight='balanced')

  # Train the model on the training data
  rf_model.fit(X_train, Y_train)

  print("Random Forest model trained successfully.")

  # Predict on the training data
  y_pred_train = rf_model.predict(X_train)
  y_pred_proba_train = rf_model.predict_proba(X_train)[:, 1]

  # Predict on the test data
  y_pred_test = rf_model.predict(X_test)
  y_pred_proba_test = rf_model.predict_proba(X_test)[:, 1]

  # Calculate evaluation metrics
  from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

  # Training Set Metrics
  auc_train = roc_auc_score(Y_train, y_pred_proba_train)
  f1_train = f1_score(Y_train, y_pred_train)
  precision_train = precision_score(Y_train, y_pred_train)
  recall_train = recall_score(Y_train, y_pred_train)

  # Test Set Metrics
  auc_test = roc_auc_score(Y_test, y_pred_proba_test)
  f1_test = f1_score(Y_test, y_pred_test)
  precision_test = precision_score(Y_test, y_pred_test)
  recall_test = recall_score(Y_test, y_pred_test)

  results = {
      'Fold': fold_id,
      'Model': 'Random Forest',
      'Parameters': f'max_depth={d}, n_estimators={n}',
      'ROC_AUC_Train': auc_train,
      'F1_Score_Train': f1_train,
      'Precision_Train': precision_train,
      'Recall_Train': recall_train,
      'ROC_AUC_Test': auc_test,
      'F1_Score_Test': f1_test,
      'Precision_Test': precision_test,
      'Recall_Test': recall_test
  }

  return results

#XGBoost

In [ ]:
def xgboost(fold_id, X_train, Y_train, X_test, Y_test):
  """
    Train an XGBoost model and evaluate its performance.

    Args:
        fold_id (int): The fold number for logging.
        X_train (pd.DataFrame): Features for the training set.
        Y_train (pd.Series): Target variable for the training set.
        X_test (pd.DataFrame): Features for the testing set.
        Y_test (pd.Series): Target variable for the testing set.

    Returns:
        dict: A dictionary containing ROC_AUC, F1-Score, Precision, and Recall
              for both training and testing data.
  """
  import xgboost as xgb
  from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

  # Initialize the XGBoost model
  xgb_model = xgb.XGBClassifier(
      objective='binary:logistic',
      eval_metric='logloss',
      use_label_encoder=False, # Suppress the warning
      random_state=42,
      n_estimators=100,
      max_depth=5,
      scale_pos_weight=(Y_train.value_counts()[False] / Y_train.value_counts()[True]) # Handle class imbalance
  )

  # Train the model on the training data
  xgb_model.fit(X_train, Y_train)

  print("XGBoost model trained successfully.")

  # Predict on the training data
  y_pred_train = xgb_model.predict(X_train)
  y_pred_proba_train = xgb_model.predict_proba(X_train)[:, 1]

  # Predict on the test data
  y_pred_test = xgb_model.predict(X_test)
  y_pred_proba_test = xgb_model.predict_proba(X_test)[:, 1]

  # Training Set Metrics
  auc_train = roc_auc_score(Y_train, y_pred_proba_train)
  f1_train = f1_score(Y_train, y_pred_train)
  precision_train = precision_score(Y_train, y_pred_train)
  recall_train = recall_score(Y_train, y_pred_train)

  # Test Set Metrics
  auc_test = roc_auc_score(Y_test, y_pred_proba_test)
  f1_test = f1_score(Y_test, y_pred_test)
  precision_test = precision_score(Y_test, y_pred_test)
  recall_test = recall_score(Y_test, y_pred_test)

  results = {
      'Fold': fold_id,
      'Model': 'XGBoost',
      'Parameters': 'n_estimators=100, max_depth=5',
      'ROC_AUC_Train': auc_train,
      'F1_Score_Train': f1_train,
      'Precision_Train': precision_train,
      'Recall_Train': recall_train,
      'ROC_AUC_Test': auc_test,
      'F1_Score_Test': f1_test,
      'Precision_Test': precision_test,
      'Recall_Test': recall_test
  }

  return results

In [ ]:
# 用于存储所有 fold 的评估结果
all_fold_results = []

# 假设你的 train_folds 和 test_folds 已经生成好，并且长度为 3
for i, (train_f, test_f) in enumerate(zip(train_folds, test_folds)):
    fold_id = i + 1
    print(f"\n==================== Running Fold {fold_id} ====================")

    df_train_fold = create_complete_dataset_for_modeling(train_f, df_multiple_visits, df_withID_no_cancel);

    #checkout NaNs in df_train_fold


    #df_multiple_visits_test = remove_one_hit_wonders(df_visit)
    df_test_fold = create_complete_dataset_for_modeling(test_f, df_multiple_visits, df_withID_no_cancel);

    # 3. 准备 X 和 y (训练集 & 测试集)
    # 假设特征列为除最后两列之外的所有列
    feature_cols = df_train_fold.columns[:-2].tolist();

    X_train = df_train_fold[feature_cols];
    y_train = df_train_fold['Churn'];

    # 测试集需要对齐特征列（防止有缺失或不一致）
    X_test = df_test_fold[feature_cols];
    y_test = df_test_fold['Churn'];




    # 5. 模型训练 (以 XGBoost 或 Random Forest 为例)
    # 你也可以换成 LogisticRegression()
    # model = RandomForestClassifier(random_state=42, max_depth=5, n_estimators=100, class_weight='balanced')
    # model.fit(X_train, y_train)

    # 6. 在测试集上进行预测
    # y_pred = model.predict(X_test)
    # y_pred_proba = model.predict_proba(X_test)[:, 1] # 获取预测为 Churn 的概率

    # 7. 计算评估指标
    # from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

    # fold_auc = roc_auc_score(y_test, y_pred_proba)
    # fold_f1 = f1_score(y_test, y_pred)
    # fold_precision = precision_score(y_test, y_pred)
    # fold_recall = recall_score(y_test, y_pred)

    # 8. 将当前 Fold 的结果保存到列表
    # all_fold_results.append({
    #     'Fold': fold_id,
    #     'ROC_AUC': fold_auc,
    #     'F1_Score': fold_f1,
    #     'Precision': fold_precision,
    #     'Recall': fold_recall
    # })

    all_fold_results.append(logistic_regression(fold_id, X_train, y_train, X_test, y_test, 0.001));
    all_fold_results.append(logistic_regression(fold_id, X_train, y_train, X_test, y_test, 0.01));
    all_fold_results.append(logistic_regression(fold_id, X_train, y_train, X_test, y_test, 0.1));
    all_fold_results.append(logistic_regression(fold_id, X_train, y_train, X_test, y_test, 1));
    all_fold_results.append(logistic_regression(fold_id, X_train, y_train, X_test, y_test, 10));
    all_fold_results.append(random_forest(fold_id, X_train, y_train, X_test, y_test, 5, 100));
    all_fold_results.append(random_forest(fold_id, X_train, y_train, X_test, y_test, 6, 100));

# ==================== 汇总与统计 ====================
results_df = pd.DataFrame(all_fold_results);
print("\n--- All Folds Evaluation Summary ---");
display(results_df);

print("\n--- Mean & Standard Deviation of Metrics by Model ---");
metrics_to_agg = [
    'ROC_AUC_Train', 'F1_Score_Train', 'Precision_Train', 'Recall_Train',
    'ROC_AUC_Test', 'F1_Score_Test', 'Precision_Test', 'Recall_Test'
];
mean_std_metrics = results_df.groupby('Model')[metrics_to_agg].agg(['mean', 'std']);
display(mean_std_metrics.round(3));


==================== Running Fold 1 ====================
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Random Forest model trained successfully.
Random Forest model trained successfully.

==================== Running Fold 2 ====================
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Random Forest model trained successfully.
Random Forest model trained successfully.

==================== Running Fold 3 ====================
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained successfully.
Logistic Regression model trained 

,Fold,Model,Parameters,ROC_AUC_Train,F1_Score_Train,Precision_Train,Recall_Train,ROC_AUC_Test,F1_Score_Test,Precision_Test,Recall_Test
0,1,Logistic Regression,C=0.001,0.724302,0.620054,0.519974,0.767840,0.746697,0.715044,0.624420,0.836439
1,1,Logistic Regression,C=0.01,0.730944,0.627970,0.531662,0.766889,0.750813,0.718889,0.628749,0.839199
2,1,Logistic Regression,C=0.1,0.735604,0.632124,0.543896,0.754520,0.753071,0.717887,0.635157,0.825397
3,1,Logistic Regression,C=1,0.736091,0.630400,0.543823,0.749762,0.752984,0.716049,0.635150,0.820566
4,1,Logistic Regression,C=10,0.736093,0.631958,0.545643,0.750714,0.752886,0.716094,0.635634,0.819876
5,1,Random Forest,"max_depth=5, n_estimators=100",0.800641,0.673142,0.605163,0.758325,0.753416,0.680697,0.650126,0.714286
6,1,Random Forest,"max_depth=6, n_estimators=100",0.829374,0.696170,0.629715,0.778306,0.751781,0.676637,0.652564,0.702553
7,2,Logistic Regression,C=0.001,0.722795,0.594616,0.474116,0.797240,0.736205,0.655057,0.556325,0.796396
8,2,Logistic Regression,C=0.01,0.726248,0.599601,0.480486,0.797240,0.739779,0.663714,0.562226,0.809910
9,2,Logistic Regression,C=0.1,0.727214,0.602116,0.488127,0.785563,0.740183,0.664417,0.569043,0.798198



--- Mean & Standard Deviation of Metrics by Model ---


ROC_AUC_Train        F1_Score_Train         \
                             mean    std           mean    std   
Model                                                            
Logistic Regression         0.729  0.004          0.611  0.013   
Random Forest               0.809  0.017          0.669  0.018   

                    Precision_Train        Recall_Train        ROC_AUC_Test  \
                               mean    std         mean    std         mean   
Model                                                                         
Logistic Regression            0.51  0.024        0.765  0.024        0.744   
Random Forest                  0.58  0.033        0.793  0.021        0.752   

                           F1_Score_Test        Precision_Test         \
                       std          mean    std           mean    std   
Model                                                                   
Logistic Regression  0.006         0.668  0.040          0.573  0.047   
Random Forest        0.001         0.659  0.021          0.595  0.050   

                    Recall_Test         
                           mean    std  
Model                                   
Logistic Regression       0.801  0.027  
Random Forest             0.742  0.028